Online Store Database Engineering and Analytics Project

Project Overview
This project demonstrates the design and implementation of a relationalndatabase for an online store using PostgreSQL.
The project focuses on database design, data modeling, data integrity, SQL transformations, analytical queries, and database performance concepts.

The project uses PostgreSQL running in Docker, with DBeaver used for database management and Metabase used for SQL-based analysis and data visualization.

Technologies
- PostgreSQL 16
- Docker
- Python
- Jupyter Notebook
- DBeaver
- SQL
- Metabase
- Git and Github



-- Project Overview
The objective of this project was to build and analyze a relational database for an online store.
The project was designed as a practical Data Engineering learning project to develop skills in:
- Relational database design
- PostgreSQL
- SQL
- Data modeling
- Data integrity and constraints
- Docker
- DBeaver
- SQL-based analytics
- Data visualization using Metabase
- Python and PostgreSQL integration

The database stores information about product categories, customers, products, orders, order items, and customer reviews.

The project follows the workflow:
Database Design → Schema Creation → Data Population → Data Validation → SQL Analysis → Visualization

Objectives
1. Design a normalized relational database.
2. Deploy PostgreSQL using Docker.
3. Create and populate database tables.
4. Implement relationships and data integrity constraints.
5. Perform analytical queries using SQL.
6. Explore indexes, views, and materialized views.
7. Connect Python to PostgreSQL.

Project Workflow

Python Environment - Setted up the python env to provide an isolated workspace where I can write the Python code and install project-specific packages without affecting my computer's global Python setup.
       ↓
Docker - Created and run a consistent and isolated environment for PostgreSQL database without needing to install it directly on my computer
       ↓
PostgreSQL - The relational database system where the online store data is stored, organized, and managed.
       ↓
Database Schema - Defined the structure of the database; tables, columns, data types, primary keys, foreign keys, and constraints.
       ↓
Sample Data - Populated the tables with records to get actual data to work with and test your database.
       ↓
SQL Queries/ Dbeaver - Retrieved, filtered, joined, aggregated, and transformed the data to answer questions about the online store.
       ↓
Analytical Results - Metabase
       ↓
Future ETL Pipeline - Automate the process of Extracting data from sources, Transforming/cleaning it, and Loading it into your database or analytical system.


Project Architecture
VS Code
- schema.sql
- data.sql
- SQL scripts
        ↓
Docker
        ↓
PostgreSQL
        ↓
online_store database
        ↓
DBeaver
        ↓
Metabase
        ↓
Analysis & Visualization

In [ ]:
-- 1. Environment Setup
-- creating a python virtual environment to isolate project dependencies from the system Python installation.

python -m venv venv 

In [ ]:
--2. Deploying PostgreSQL
-- deploying PostgreSQL using Docker so that the database environment could be reproduced independently of the 
-- local operating system.

docker run --name my-postgres-study \
-e POSTGRES_USER=student \
-e POSTGRES_PASSWORD=<PASSWORD> \
-e POSTGRES_DB=online_store_c \
-p 5433:5432 \
-d postgres:16

In [ ]:
-- 2.b Connecting Database
-- Using Python to connect PostgreSQL database running inside the Docker container
-- The connection allows database operations to be executed programmatically rather than relying entirely on a database GUI.

import psycopg2

connection = psycopg2.connect(
    host="localhost",
    port=5433,
    database="online_store_c",
    user="student",
    password=****("POSTGRES_PASSWORD")
)

print("Database connection successful")

In [ ]:

After successful database connection, the tables were created. Instead of directly pasting the long query in the postgres 
database using the terminal, I created a SQL file (schema.sql) to contain all the query and then run it in docker using:

- docker exec -i my-postgres psql -U student -d online_store_c < schema.sql

Advantages of this method:
1. Clean and organized
2. Easy to edit and reuse
3. Version control (very important in real projects)
4. Works for large schemas

So workflow  becomes:
1. Write schema → schema.sql
2. Run it → Docker + psql
3. Query → GUI or terminal


-- 3. Database Schema Design
The database was designed using a normalized relational model.

The main entities are:

- Categories
- Customers
- Products
- Orders
- Order Items
- Reviews

Primary and foreign keys were implemented to maintain relationships
between entities and preserve referential integrity.


ERD :
categories
    │
    └── products
            │
            └── order_items
                    │
orders ──────────────┘
  │
  └── customers


In [ ]:
-- 4. Schema creation - using notebook
Designed the relational database schema and created the required tables, relationships, and constraints.
Schema can be found in a seperate file in the folder.

But Here are a few schema design decisions:

for customers table-
- `customer_id` acts as the primary key.
- `email` is unique to prevent duplicate customer accounts.
- `NOT NULL` is applied to required fields.
- `country` defaults to Ghana (`GH`).
- Timestamp columns automatically record creation and update times.

for category table-
`category_id` acts as the primary key.
- `name` and `slug` are unique to prevent duplicate categories.
- `parent_id` references the same table, allowing categories to have subcategories.
- `is_active` indicates whether a category is currently available.
- `created_at` automatically records when the category was created.

for products table-
- `product_id` acts as the primary key.
- `category_id` links each product to its category using a foreign key.
- `sku` is unique to give each product a distinct stock-keeping identifier.
- `NOT NULL` is applied to required fields such as category, SKU, name, price, and stock quantity.
- `price` and `cost` have CHECK constraints to prevent negative values.
- `stock_qty` defaults to 0 and cannot be negative.
- `is_active` indicates whether a product is currently available for sale.
- Timestamp columns record when the product was created and last updated.

for order table-
`order_id` acts as the primary key.
- `customer_id` links each order to the customer who placed it.
- `status` tracks the current stage of the order.
- A CHECK constraint restricts status values to predefined options such as pending, confirmed, shipped, delivered, cancelled, and refunded.
- `total_amount` cannot be negative.
- `shipping_fee` and `discount_amount` default to 0.00 when no value is provided.
- `ordered_at` automatically records when the order was created.
- `shipped_at` and `delivered_at` can be NULL because an order may not have been shipped or delivered yet.

for orders table-
- `order_item_id` acts as the primary key.
- `order_id` links each item to its corresponding order.
- `product_id` links each item to the product that was purchased.
- `quantity` must be greater than zero.
- `unit_price` records the product's price at the time of purchase.
- `discount_pct` defaults to 0.00 and is restricted to values between 0 and 100.
- A UNIQUE constraint on `order_id` and `product_id` prevents the same product from appearing more than once in a single order.

for review table-
- `review_id` acts as the primary key.
- `product_id` identifies the product being reviewed.
- `customer_id` identifies the customer who submitted the review.
- `rating` is restricted to values from 1 to 5 using a CHECK constraint.
- `is_verified` indicates whether the review comes from a verified purchase.
- `created_at` automatically records when the review was submitted.
- A UNIQUE constraint on `product_id` and `customer_id` ensures that a customer can submit only one review for each product.


Summary of the Data Quality Rules used - 

| Data Quality Rule          | Example                                                      | Purpose                                                      |
| -------------------------- | ------------------------------------------------------------ | ------------------------------------------------------------ |
|   1. Primary keys          | `customer_id`, `product_id`, `order_id`                      | Ensures every record has a unique identifier                 |
|   2. Foreign keys          | `products.category_id → categories.category_id`              | Prevents records from referencing non-existent data          |
|   3. NOT NULL              | Customer email, product name, order customer                 | Ensures required fields are always populated                 |
|   4. UNIQUE                | Customer email, product SKU                                  | Prevents duplicate values where uniqueness is required       |
|   5. CHECK constraints     | `price >= 0`, `quantity > 0`                                 | Prevents invalid values                                      |
|   6. Range validation      | `rating BETWEEN 1 AND 5`                                     | Keeps values within an acceptable range                      |
|   7. Default values        | `stock_qty = 0`, `discount_pct = 0`                          | Provides sensible values when none are supplied              |
|   8. Controlled values     | Order status must be `pending`, `confirmed`, `shipped`, etc. | Prevents inconsistent or invalid categories                  |
|   9. Relationship integrity| Orders must reference an existing customer                   | Maintains consistency between related tables                 |
|   10.Duplicate prevention  | `UNIQUE(order_id, product_id)`                               | Prevents the same product from being added twice to an order |
|   11.Timestamp tracking    | `created_at`, `ordered_at`, `updated_at`                     | Helps track when records were created or modified            |


In [ ]:
-- Now schema has been created and connected to the online_store_c database.

The table is now ready to receive data 

In [ ]:
6. Data Population

After creating the database tables, I populated the online_store database with sample data using SQL INSERT statements.

I created a separate sample_data.sql file in VS Code to organize the data-population queries. The file contained INSERT INTO statements for each of the database tables.

** sample_data.sql file can be found in the folder

--Data Loading Workflow
1. Created the database tables and constraints in PostgreSQL.
2. Prepared sample records in sample_data.sql using INSERT INTO statements.
3. Executed the SQL statements against the PostgreSQL online_store database through the Terminal
4. Loaded data into the parent tables before the dependent tables to satisfy foreign-key constraints.
5. Validated the number of records loaded into each table.
6. Troubleshot and corrected constraint, column, and foreign-key errors encountered during the loading process.

--LOading Order
Because the database uses foreign keys, the tables were populated in an order that respected their dependencies:

categories
    ↓
products

customers
    ↓
orders
    ↓
order_items

customers + products
    ↓
reviews

For example, products reference categories, so the categories needed to exist before products could be inserted. Similarly, order_items references both orders and products, so those records needed to exist before order items could be loaded.

--Data Intergrity during loading
The database constraints helped identify invalid or inconsistent records during the population process. Errors encountered during loading included foreign-key violations, missing required values, and mismatches between the SQL data and table definitions.
These issues were corrected before validating the final dataset.

--Validation
After loading the data, I used SQL queries to verify the number of records in each table:
SELECT 'categories' AS table_name, COUNT(*) FROM categories
UNION ALL
SELECT 'customers', COUNT(*) FROM customers
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'reviews', COUNT(*) FROM reviews;

This provided a simple data-loading validation check and helped confirm that the expected records were present in each table.


--Key Learning
This stage demonstrated that data population is not simply about inserting records. The loading process must respect table dependencies, foreign-key relationships, data types, and database constraints. 
Separating the population queries into data.sql also made the data-loading process easier to organize, reproduce, and troubleshoot.




In [ ]:
--Analytics
Metabase was connected to the PostgreSQL database to explore the data and create analytical visualizations.
Metabase was connected directly to the PostgreSQL online_store_c database running in Docker.
The connection allowed the populated database tables to be queried directly from Metabase without creating a separate copy of the data.
The analytical SQL queries were then saved as questions in Metabase.
Metabase was used to explore the results and determine which metrics were most appropriate for a management dashboard.

SQL Analysis
After validating the populated database, analytical queries were developed to answer business questions about sales, customers, products, orders, and reviews.

The analysis consisted of 15 questions covering:

Sales & Orders
- Sales overview
- Order status distribution
- Payment method usage
- Revenue by city

Products & Categories
- Product revenue
- Category revenue
- Sales volume
- Cheapest products by category
- Product profit margins

Customers
- Top customers by spending
- Orders per customer
- Customers without orders

Reviews
- Product ratings
- Relationship between ratings and sales

-- Check `Metabase Analytics and Visualization.ipynb` for detailed analytics and Dashboard



 -- Challenges and Lessons Learned

During the project, several technical challenges were encountered.

1. Docker and PostgreSQL Connectivity
I learned how PostgreSQL can be deployed inside a Docker container and accessed by external tools such as DBeaver and Metabase.
I also learned the difference between the PostgreSQL container port (`5432`) and the host-mapped port (`5433`).

2. Foreign Key Constraints
Data loading initially produced foreign-key errors when parent records had not been successfully inserted.
This reinforced the importance of loading related tables in the correct dependency order.

3. Database Constraints
Several data-loading errors demonstrated how NOT NULL, CHECK, UNIQUE, and FOREIGN KEY constraints protect database integrity.

4. Metabase Connectivity
Connecting Metabase to PostgreSQL required understanding how containers communicate through Docker networking.

5. Metric Definition
The revenue discrepancy between order-level totals and line-item calculations highlighted the importance of consistent metric definitions in analytics.

-- Future Improvements
Future versions of this project could include:

- Building a Python ETL pipeline for automated data ingestion.
- Connecting Python directly to PostgreSQL.
- Implementing incremental data loading.
- Adding logging and error handling.
- Using dbt for data transformation and testing.
- Using Airflow to orchestrate the pipeline.
- Adding automated data-quality tests.
- Creating historical sales analysis using larger datasets.
- Adding inventory forecasting.
- Improving customer segmentation.
- Containerizing the complete application stack using Docker Compose.